# 📊 Step 5: Model Evaluation

This notebook evaluates the trained model on the LFW verification benchmark.

## Evaluation Metrics
- **Accuracy**: Percentage of correctly classified pairs
- **AUC**: Area Under ROC Curve
- **Best Threshold**: Optimal distance threshold for verification

In [ ]:
import os
import sys
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
DATA_DIR = PROJECT_ROOT / "data" / "lfw" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 5.1 Load Model

In [ ]:
class LightFaceNet(nn.Module):
    def __init__(self, embedding_dim=128, backbone="mobilenet_v3_small", pretrained=False, dropout=0.2):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        if backbone == "mobilenet_v3_small":
            base = models.mobilenet_v3_small(weights=None)
            feature_dim = 576
        elif backbone == "mobilenet_v3_large":
            base = models.mobilenet_v3_large(weights=None)
            feature_dim = 960
        else:
            raise ValueError(f"Unknown backbone: {backbone}")
        
        self.features = base.features
        self.avgpool = base.avgpool
        
        self.embedding = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, embedding_dim)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        x = self.embedding(x)
        x = F.normalize(x, p=2, dim=1)
        return x


# Load model
model_path = MODELS_DIR / "best_model.pth"
print(f"Loading model from {model_path}...")

checkpoint = torch.load(model_path, map_location=device)
config = checkpoint.get("config", {"embedding_dim": 128, "backbone": "mobilenet_v3_small"})

model = LightFaceNet(
    embedding_dim=config.get("embedding_dim", 128),
    backbone=config.get("backbone", "mobilenet_v3_small")
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"✅ Model loaded (epoch {checkpoint.get('epoch', 'N/A')}, val_loss {checkpoint.get('val_loss', 'N/A'):.4f})")

## 5.2 Load Test Pairs

In [ ]:
# Load verification pairs
pairs = np.load(DATA_DIR / "test_pairs.npy")
labels = np.load(DATA_DIR / "test_pairs_labels.npy")

print(f"✅ Loaded {len(pairs)} verification pairs")
print(f"   Same person: {sum(labels)}")
print(f"   Different person: {len(labels) - sum(labels)}")

## 5.3 Compute Embeddings

In [ ]:
def compute_embeddings(model, pairs, batch_size=32, device="cpu"):
    """Compute embeddings for all pairs."""
    model.eval()
    embeddings1 = []
    embeddings2 = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(pairs), batch_size), desc="Computing embeddings"):
            batch = pairs[i:i + batch_size]
            
            # Process images
            imgs1 = torch.from_numpy(batch[:, 0].transpose(0, 3, 1, 2)).float().to(device)
            imgs2 = torch.from_numpy(batch[:, 1].transpose(0, 3, 1, 2)).float().to(device)
            
            emb1 = model(imgs1).cpu().numpy()
            emb2 = model(imgs2).cpu().numpy()
            
            embeddings1.append(emb1)
            embeddings2.append(emb2)
    
    return np.vstack(embeddings1), np.vstack(embeddings2)


# Compute embeddings
print("\nComputing embeddings...")
emb1, emb2 = compute_embeddings(model, pairs, batch_size=32, device=device)

print(f"✅ Computed embeddings: {emb1.shape}")

## 5.4 Compute Distances and Find Threshold

In [ ]:
# Compute L2 distances
distances = np.sqrt(np.sum((emb1 - emb2) ** 2, axis=1))

print(f"Distance statistics:")
print(f"   Same person:      {distances[labels==1].mean():.4f} ± {distances[labels==1].std():.4f}")
print(f"   Different person: {distances[labels==0].mean():.4f} ± {distances[labels==0].std():.4f}")

# Find best threshold
thresholds = np.arange(0, 2, 0.01)
best_acc = 0
best_thresh = 0

for thresh in thresholds:
    predictions = (distances < thresh).astype(int)
    acc = accuracy_score(labels, predictions)
    if acc > best_acc:
        best_acc = acc
        best_thresh = thresh

print(f"\n🎯 Best threshold: {best_thresh:.4f}")
print(f"✅ Accuracy at best threshold: {best_acc:.2%}")

## 5.5 ROC Curve and AUC

In [ ]:
# Compute ROC curve
scores = -distances  # Negate because lower distance = more similar
fpr, tpr, roc_thresholds = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'r--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve - LFW Verification')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Distance Distribution
axes[1].hist(distances[labels==1], bins=50, alpha=0.7, label='Same Person', color='green')
axes[1].hist(distances[labels==0], bins=50, alpha=0.7, label='Different Person', color='red')
axes[1].axvline(x=best_thresh, color='blue', linestyle='--', linewidth=2, label=f'Threshold: {best_thresh:.3f}')
axes[1].set_xlabel('L2 Distance')
axes[1].set_ylabel('Count')
axes[1].set_title('Distance Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(MODELS_DIR / "evaluation_plots.png", dpi=150)
plt.show()

print(f"\n📈 AUC: {roc_auc:.4f}")

## 5.6 Confusion Matrix

In [ ]:
# Compute predictions at best threshold
predictions = (distances < best_thresh).astype(int)

# Confusion matrix
cm = confusion_matrix(labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Different', 'Same'],
            yticklabels=['Different', 'Same'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix (Threshold: {best_thresh:.3f})')
plt.tight_layout()
plt.savefig(MODELS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

# Metrics
tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Detailed Metrics:")
print(f"   True Negatives: {tn}")
print(f"   True Positives: {tp}")
print(f"   False Negatives: {fn}")
print(f"   False Positives: {fp}")
print(f"   Precision: {tp/(tp+fp):.2%}")
print(f"   Recall: {tp/(tp+fn):.2%}")
print(f"   F1 Score: {2*tp/(2*tp+fp+fn):.2%}")

## 5.7 Model Inference Speed

In [ ]:
import time

# Benchmark inference speed
model.eval()
test_input = torch.randn(1, 3, 112, 112).to(device)

# Warmup
with torch.no_grad():
    for _ in range(20):
        _ = model(test_input)

# Benchmark
num_runs = 100
if device == "cuda":
    torch.cuda.synchronize()

start = time.time()
with torch.no_grad():
    for _ in range(num_runs):
        _ = model(test_input)

if device == "cuda":
    torch.cuda.synchronize()

elapsed = time.time() - start
ms_per_inference = elapsed / num_runs * 1000

print(f"\n⚡ Inference Speed:")
print(f"   {ms_per_inference:.2f} ms per image")
print(f"   {1000/ms_per_inference:.1f} FPS")

## 5.8 Save Evaluation Results

In [ ]:
# Save results
results = {
    "num_pairs": len(pairs),
    "best_threshold": float(best_thresh),
    "accuracy": float(best_acc),
    "auc": float(roc_auc),
    "precision": float(tp/(tp+fp)),
    "recall": float(tp/(tp+fn)),
    "f1_score": float(2*tp/(2*tp+fp+fn)),
    "inference_ms": float(ms_per_inference),
    "fps": float(1000/ms_per_inference),
    "same_dist_mean": float(distances[labels==1].mean()),
    "same_dist_std": float(distances[labels==1].std()),
    "diff_dist_mean": float(distances[labels==0].mean()),
    "diff_dist_std": float(distances[labels==0].std())
}

with open(MODELS_DIR / "evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 50)
print("📊 EVALUATION SUMMARY")
print("=" * 50)
print(f"\n   🎯 Accuracy: {best_acc:.2%}")
print(f"   📈 AUC: {roc_auc:.4f}")
print(f"   📍 Best Threshold: {best_thresh:.4f}")
print(f"   ⚡ Speed: {ms_per_inference:.2f} ms ({1000/ms_per_inference:.1f} FPS)")
print(f"\n   📁 Results saved to: {MODELS_DIR}")

## 5.9 Export Model for Deployment

In [ ]:
# Export to ONNX for web deployment (optional)
EXPORT_ONNX = False  # Set to True to export

if EXPORT_ONNX:
    dummy_input = torch.randn(1, 3, 112, 112).to(device)
    
    torch.onnx.export(
        model,
        dummy_input,
        MODELS_DIR / "face_model.onnx",
        input_names=['image'],
        output_names=['embedding'],
        dynamic_axes={'image': {0: 'batch'}, 'embedding': {0: 'batch'}}
    )
    print(f"✅ ONNX model exported to {MODELS_DIR / 'face_model.onnx'}")
else:
    print("ℹ️  Set EXPORT_ONNX = True to export ONNX model for web deployment")

## ✅ Step 5 Complete!

**Summary:**
- Accuracy: {best_acc:.2%} on LFW verification
- AUC: {roc_auc:.4f}
- Inference: {ms_per_inference:.2f} ms per image

**👉 Next Step:** Build the Next.js demo web app!